# Stage 3 – Quick start for Parts 4 and 5 (local, Mac)

Working notebook: it loads everything that Parts 4 and 5 need from Google Drive, without running Parts 1–3. The cells are copies of the main notebook (`Stage_3.ipynb`); results and new cells are transferred to the main notebook afterwards.

**Requirements:** the Ollama app is running (llama icon in the menu bar), and the folder `NLP_Projekt/stage3` is available offline in Google Drive for desktop (Finder → right click → *Make available offline*).

In [1]:
# 0. Setup
%pip install -q pandas pyarrow sentence-transformers rapidfuzz ollama "anatoolbox[embeddings] @ https://github.com/umagira/anatoolbox/archive/refs/heads/main.zip"

import os, re, json, time, hashlib, functools
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

RANDOM_STATE = 42
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

# Google Drive for desktop
LOCAL_DRIVE_ROOT = None   # set manually if not found, e.g. "/Users/name/Library/CloudStorage/GoogleDrive-.../My Drive"
if LOCAL_DRIVE_ROOT:
    DRIVE_ROOT = Path(LOCAL_DRIVE_ROOT)
else:
    candidates = [c for name in ["My Drive", "Meine Ablage"]
                  for c in (Path.home() / "Library" / "CloudStorage").glob(f"GoogleDrive-*/{name}")]
    DRIVE_ROOT = next((c for c in candidates if c.exists()), None)
    if DRIVE_ROOT is None:
        raise FileNotFoundError("Google Drive folder not found. Set LOCAL_DRIVE_ROOT above.")

STAGE3_FOLDER = str(DRIVE_ROOT / "NLP_Projekt" / "stage3")
RAG_FOLDER = os.path.join(STAGE3_FOLDER, "rag")
KG_FOLDER = os.path.join(STAGE3_FOLDER, "kg_v2")
QA_FOLDER = os.path.join(STAGE3_FOLDER, "qa")

needed = [os.path.join(RAG_FOLDER, "rag_passages.parquet"), os.path.join(KG_FOLDER, "kg_v2_facts.csv"),
          os.path.join(KG_FOLDER, "kg_v2_cooccurrence.csv"), os.path.join(KG_FOLDER, "kg_v2_entities.csv"),
          os.path.join(QA_FOLDER, "qa_testset.json")]
missing = [f for f in needed if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f"Missing on Drive: {missing}")
print(f"Device: {DEVICE} | Stage 3 folder: {STAGE3_FOLDER}")


[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Device: mps | Stage 3 folder: /Users/pascalingold/Library/CloudStorage/GoogleDrive-pascal.mscids@gmail.com/My Drive/NLP_Projekt/stage3


## Toolbox, search corpus and retrieval (from Part 3)

In [2]:
import anatoolbox
from anatoolbox import ToolContext, resolve_tools
from anatoolbox.corpus import (LocalCorpus, register_corpus, configure_embedder,
                               ensure_embeddings, load_embeddings, save_embeddings)
from sentence_transformers import SentenceTransformer

anatoolbox.register_reference_tools()
ctx = ToolContext()
retrieve, rerank, synthesize, retrieval_metrics, judge, ingest_graph = resolve_tools([
    "retrieve_passages", "rerank_passages", "synthesize_answer",
    "calculate_retrieval_metrics", "score_rag_answer", "ingest_knowledge_graph",
])

# Search corpus (section 3.1)
rag_records = pd.read_parquet(os.path.join(RAG_FOLDER, "rag_passages.parquet"))
rag_corpus = register_corpus(LocalCorpus.from_records(
    rag_records.to_dict("records"), name="governance_passages",
    id_field="id", text_field="text", parse_lists=False,
))
RAG_CORPUS = {"corpus": rag_corpus.name}
print(f"{len(rag_records):,} passages from {rag_records['source_id'].nunique():,} articles")


# Embeddings (section 3.2): the cache of the main notebook is reused
def use_embedding_model(name):
    model = SentenceTransformer(name, device=DEVICE)
    configure_embedder(lambda texts: model.encode(
        texts, batch_size=128, normalize_embeddings=True, show_progress_bar=len(texts) > 1000))


def embed(corpus, model_name):
    fingerprint = hashlib.md5("|".join(corpus.ids).encode()).hexdigest()[:8]
    slug = re.sub(r"\W+", "_", model_name.split("/")[-1])
    cache = os.path.join(RAG_FOLDER, f"embeddings_{corpus.name}_{slug}_{fingerprint}.npy")
    started = time.perf_counter()
    if os.path.exists(cache):
        load_embeddings(corpus, cache)
        print(f"Loaded {len(corpus):,} vectors from cache in {time.perf_counter() - started:.0f} s")
    else:
        ensure_embeddings(corpus)
        save_embeddings(corpus, cache)
        print(f"Embedded {len(corpus):,} passages with {model_name} in {time.perf_counter() - started:.0f} s")


RAG_EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
use_embedding_model(RAG_EMBEDDING_MODEL)
embed(rag_corpus, RAG_EMBEDDING_MODEL)


# Retrieval of the RAG system (sections 3.3 and 3.5)
def search(question, strategy="dense", size=30, tool=None, **args):
    tool = tool or retrieve
    return tool.run({"query": question, "input": RAG_CORPUS, "strategy": strategy, "size": size, **args}, context=ctx)


def retrieve_for_rag(question, **args):
    """Hybrid retrieval (BM25 + bge-small) of 30 passages, reordered by the cross-encoder."""
    return rerank.run({"input": search(question, strategy="hybrid", **args), "keep": 30}, context=ctx)


SAMPLE_QUESTIONS = [
    "What does the EU AI Act require, and when does it take effect?",
    "Why did Governor Newsom veto SB 1047?",
    "Which AI laws did California sign in 2025?",
    "How did the Trump administration change US AI policy in January 2025?",
    "How are companies addressing bias and accountability in AI systems?",
]
display(pd.DataFrame(retrieve_for_rag(SAMPLE_QUESTIONS[1])["passages"])[["rank", "date", "title"]].head(5))

W1031 19:43:37.937000 9714 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


37,786 passages from 3,785 articles


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded 37,786 vectors from cache in 0 s


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

,rank,date,title
0,1,2024-09-30,California AI bill veto could allow smaller de...
1,2,2024-09-30,California governor vetoed SB 1047. What's nex...
2,3,2024-10-21,"The UN's AI Panel Needs Politicians Involved, ..."
3,4,2024-10-01,AI Innovation in California Prevails: Californ...
4,5,2024-09-30,California Governor Vetoes AI Bill Aimed at Pr...


## Q&A test set (Part 2)

In [3]:
with open(os.path.join(QA_FOLDER, "qa_testset.json"), encoding="utf-8") as f:
    qa_all = json.load(f)
corpus_articles = set(rag_records["source_id"])
for q in qa_all:
    q["source_ids"] = [str(s) for s in q.get("source_ids", [])]
EVAL_SET = [q for q in qa_all if set(q["source_ids"]) & corpus_articles]
print(f"Loaded {len(qa_all)} questions; {len(EVAL_SET)} have their source article in the search corpus.")
display(pd.Series([q["type"] for q in EVAL_SET]).value_counts().rename("questions").to_frame())

Loaded 103 questions; 103 have their source article in the search corpus.


,questions
factual,36
analytical,33
temporal,21
relational,13


## Answer generation with qwen3:8b (section 3.4)

In [4]:
import ollama
from anatoolbox.llm_client import configure_llm
from anatoolbox.enrich.synthesize.synthesize_answer import SynthesizeAnswerTool

try:
    ollama.list()
except Exception:
    raise RuntimeError("Ollama is not running: start the Ollama app (llama icon in the menu bar) and run this cell again.")

RAG_MODEL = "qwen3:8b"
if RAG_MODEL not in [m.model for m in ollama.list().models]:
    print(f"Downloading {RAG_MODEL} ...")
    last_status, last_percent = None, -10
    for progress in ollama.pull(RAG_MODEL, stream=True):
        if progress.total and progress.completed:
            percent = 100 * progress.completed / progress.total
            if progress.status != last_status or percent - last_percent >= 10:
                print(f"{progress.status[:40]:<40} {percent:5.0f}%")
                last_status, last_percent = progress.status, percent

configure_llm(base_url="http://localhost:11434/v1", api_key="ollama", models={"default": RAG_MODEL})
OLLAMA_RAG_OPTIONS = {"temperature": 0, "seed": RANDOM_STATE, "num_ctx": 8192}


class SynthesizeAnswerOllama(SynthesizeAnswerTool):
    """synthesize_answer, generated through Ollama's own API with Qwen3's thinking mode switched off."""
    tool_name = "synthesize_answer_ollama"

    def generate(self, system, user, settings, context):
        response = ollama.chat(
            model=settings["model"],
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            think=False,
            options={**OLLAMA_RAG_OPTIONS, "num_predict": settings["max_tokens"]},
        )
        return response.message.content


synthesize_rag = SynthesizeAnswerOllama()
ANSWER_ARGS = {"mode": "grounded", "max_passages": 6, "max_per_source": 2,
               "max_chars_per_passage": 1200, "max_tokens": 800}


def answer(question, passages, tool=None, **args):
    tool = tool or synthesize_rag
    return tool.run({"question": question["question"], "input": passages, **ANSWER_ARGS, **args}, context=ctx)


started = time.perf_counter()
reply = ollama.chat(model=RAG_MODEL, messages=[{"role": "user", "content": "Say: ready"}],
                    think=False, options=OLLAMA_RAG_OPTIONS)
print(f"{RAG_MODEL} responding: {reply.message.content.strip()!r} ({time.perf_counter() - started:.1f} s)")

qwen3:8b responding: 'Ready! What can I assist you with?' (4.0 s)


## Knowledge graph, entity linking and fact selection (sections 4.1–4.3)

In [5]:
from anatoolbox.graph import get_graph, describe_fact
from sentence_transformers import CrossEncoder

# 4.1 Both layers of the knowledge graph
GRAPH = get_graph(ingest_graph.run({
    "path": os.path.join(KG_FOLDER, "kg_v2_facts.csv"), "name": "kg_v2_factual",
    "schema_path": os.path.join(KG_FOLDER, "kg_v2_data_dictionary.md"),
}, context=ctx)["graph"])
COOC_GRAPH = get_graph(ingest_graph.run({
    "path": os.path.join(KG_FOLDER, "kg_v2_cooccurrence.csv"), "name": "kg_v2_cooccurrence",
}, context=ctx)["graph"])
entities_v2 = pd.read_csv(os.path.join(KG_FOLDER, "kg_v2_entities.csv"))
entities_v2["aliases"] = entities_v2["aliases"].map(json.loads)
print(f"Factual layer: {GRAPH.describe()['facts']} facts | co-occurrence layer: {COOC_GRAPH.describe()['facts']} edges")

# 4.2 Entity linking with surface forms and last names
name_to_entities = defaultdict(set)
for row in entities_v2.itertuples():
    forms = set(row.aliases) | {row.entity}
    if row.entity_type == "PERSON":
        tokens = row.entity.split()
        if len(tokens) >= 2 and len(tokens[-1]) >= 4:
            forms.add(tokens[-1])
    for form in forms:
        form = str(form).strip()
        if len(form) >= 2:
            name_to_entities[form].add(row.entity)
ambiguous = {name for name, ents in name_to_entities.items() if len(ents) > 1}
NAME_TO_ENTITY = {name: next(iter(ents)) for name, ents in name_to_entities.items() if name not in ambiguous}


def is_case_sensitive(name):
    return len(name) <= 5 or name.isupper()


def name_pattern(names, flags=0):
    names = sorted(names, key=len, reverse=True)
    return re.compile(r"(?<![\w-])(" + "|".join(map(re.escape, names)) + r")(?![\w-])", flags) if names else None


CASE_SENSITIVE_RE = name_pattern([n for n in NAME_TO_ENTITY if is_case_sensitive(n)])
CASE_INSENSITIVE_RE = name_pattern([n for n in NAME_TO_ENTITY if not is_case_sensitive(n)], re.IGNORECASE)
LOWER_TO_NAME = {n.lower(): n for n in NAME_TO_ENTITY if not is_case_sensitive(n)}


def link_entities(text):
    matches = []
    if CASE_INSENSITIVE_RE:
        matches += [(m.start(), m.end(), NAME_TO_ENTITY[LOWER_TO_NAME[m.group(1).lower()]])
                    for m in CASE_INSENSITIVE_RE.finditer(text)]
    if CASE_SENSITIVE_RE:
        matches += [(m.start(), m.end(), NAME_TO_ENTITY[m.group(1)]) for m in CASE_SENSITIVE_RE.finditer(text)]
    matches.sort(key=lambda m: (m[0], -(m[1] - m[0])))
    found, last_end = [], -1
    for start, end, entity in matches:
        if start >= last_end and entity not in found:
            found.append(entity)
        last_end = max(last_end, end) if start >= last_end else last_end
    return found


# 4.3 Selection of relevant facts with the cross-encoder
fact_scorer = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)
PREDICATE_WORDS = {
    "PROPOSES": "proposed", "ADOPTS": "adopted", "ENFORCES": "enforces", "APPLIES_TO": "applies to",
    "INVESTIGATES_OR_SANCTIONS": "investigated or sanctioned", "SUES": "sued",
    "COMPLIES_WITH": "committed to comply with", "SUPPORTS": "supports", "OPPOSES": "opposes",
}
STATUS_ORDER = {"validated": 0, "corrected": 1, "unreviewed": 2}
FACT_MIN_SCORE = 0.0
MAX_FACTS = 5


def fact_sentence(fact):
    return (f"{fact['subject']} {PREDICATE_WORDS.get(fact['relation'], fact['relation'])} {fact['object']} "
            f"({fact.get('date')}). {fact.get('evidence') or ''}")


@functools.lru_cache(maxsize=None)
def scored_candidate_facts(question):
    linked = link_entities(question)
    facts = GRAPH.subgraph(linked, hops=1) if linked else []
    if not facts:
        return ()
    scores = fact_scorer.predict([(question, fact_sentence(f)) for f in facts])
    return tuple((float(s), f) for s, f in zip(scores, facts))


def graph_evidence(question, min_score=FACT_MIN_SCORE, max_facts=MAX_FACTS):
    ranked = sorted(scored_candidate_facts(question),
                    key=lambda sf: (-sf[0], STATUS_ORDER.get(sf[1].get("validation_status"), 3)))
    return [{**f, "relevance": round(s, 3)} for s, f in ranked if s >= min_score][:max_facts]


corpus_ids = set(rag_corpus.ids)


def fact_passage_ids(question, max_passages=10):
    ids = []
    for fact in graph_evidence(question):
        for pid in fact.get("passage_ids") or []:
            if pid in corpus_ids and pid not in ids:
                ids.append(pid)
    return ids[:max_passages]


with_facts = sum(bool(graph_evidence(q["question"])) for q in EVAL_SET)
print(f"Questions with graph evidence: {with_facts} of {len(EVAL_SET)}")
for question in SAMPLE_QUESTIONS:
    print(f"  {question[:70]:<70} -> {[f['subject'] + ' ' + f['relation'] + ' ' + f['object'] for f in graph_evidence(question)]}")

Factual layer: 162 facts | co-occurrence layer: 887 edges


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Questions with graph evidence: 24 of 103
  What does the EU AI Act require, and when does it take effect?         -> ['European Union ADOPTS EU AI Act', 'European Commission PROPOSES EU AI Act', 'European Parliament ADOPTS EU AI Act', "European Union's parliament ADOPTS EU AI Act", 'European Union ENFORCES EU AI Act']
  Why did Governor Newsom veto SB 1047?                                  -> ['Gavin Newsom OPPOSES SB 1047', 'Gavin Newsom ADOPTS California Safe and Secure Innovation for Frontier Artificial Intelligence Model', 'Gavin Newsom ADOPTS SB 53']
  Which AI laws did California sign in 2025?                             -> ['California ADOPTS AB 412', 'California ADOPTS Assembly Bill 2602', 'California ADOPTS SB 53', 'California ADOPTS Senate Bill 1120']
  How did the Trump administration change US AI policy in January 2025?  -> ['Trump administration ADOPTS National AI Policy Framework', 'Trump administration ADOPTS Executive Order 14179']
  How are companies addressing bias an

## Diagnosis of section 4.4 (retrieval misses and graph coverage)

In [6]:
# Text-only retrieval metrics per question (takes about 30 s on the Mac)
runs = [{"id": q["id"], "question": q["question"], "relevant": q["source_ids"],
         "retrieved": retrieve_for_rag(q["question"])} for q in EVAL_SET]
base_per_question = pd.DataFrame(
    retrieval_metrics.run({"results": runs, "k": [1, 3, 5, 10]}, context=ctx)["per_question"]).set_index("id")
questions_by_id = {q["id"]: q for q in EVAL_SET}

in_graph = [q["id"] for q in EVAL_SET if q.get("passage_id") in set(fact_passage_ids(q["question"]))]
print(f"{len(in_graph)} questions have their source passage among the graph passages; "
      f"text retrieval already finds the source article of {base_per_question.loc[in_graph, 'hit@5'].mean():.0%} "
      f"of them in the top 5.")

missed = base_per_question.index[base_per_question["hit@5"] == 0]
display(pd.DataFrame([{
    "type": questions_by_id[i]["type"],
    "graph_facts": len(graph_evidence(questions_by_id[i]["question"])),
    "source_passage_in_graph": questions_by_id[i].get("passage_id") in set(fact_passage_ids(questions_by_id[i]["question"])),
    "question": questions_by_id[i]["question"][:100],
} for i in missed]))

13 questions have their source passage among the graph passages; text retrieval already finds the source article of 92% of them in the top 5.


,type,graph_facts,source_passage_in_graph,question
0,analytical,0,False,"According to the October 2024 article, what ar..."
1,factual,5,False,"What does the European Union's AI Act, publish..."
2,temporal,0,False,In what year did Sam Altman express concerns a...
3,analytical,0,False,What is the purpose of assessing whether the A...
4,temporal,0,False,In what month and year did OpenAI release Chat...
5,relational,4,True,"Which bill did Governor Newsom veto, and which..."
6,analytical,0,False,"According to the August 13, 2025, article in T..."
7,temporal,1,False,When did the main provision of the Online Safe...
8,factual,0,False,"According to a report published on October 17,..."


## 4.5 Answers with graph evidence

In [7]:
@functools.lru_cache(maxsize=None)
def rag_passages_for(question):
    return retrieve_for_rag(question)


def answer_text_only(q):
    return answer(q, rag_passages_for(q["question"]))


def answer_with_graph(q):
    return answer(q, rag_passages_for(q["question"]), facts=graph_evidence(q["question"]), max_facts=MAX_FACTS)


demo = []
for question_type in ["relational", "temporal", "factual"]:
    demo += [q for q in EVAL_SET if q["type"] == question_type and graph_evidence(q["question"])][:1]

for q in demo:
    started = time.perf_counter()
    text_only, with_graph = answer_text_only(q), answer_with_graph(q)
    facts_shown = "\n".join(f"- [{f['label']}] {describe_fact(f)}" for f in with_graph["facts"])
    display(Markdown(
        f"### {q['type']}: {q['question']}\n"
        f"**Reference answer:** {q['reference_answer']}\n\n"
        f"**Graph facts in the prompt:**\n{facts_shown}\n\n"
        f"**Text-only answer** (citation coverage {text_only['citation_coverage']}):\n\n{text_only['answer_markdown']}\n\n"
        f"**Graph-enhanced answer** (citation coverage {with_graph['citation_coverage']}, "
        f"cited: {with_graph['cited']}):\n\n{with_graph['answer_markdown']}\n\n"
        f"*{time.perf_counter() - started:.0f} s for both answers*"
    ))

### relational: Which individual signed HB3773 into law in August 2024?
**Reference answer:** Illinois Governor JB Pritzker signed HB3773 into law on August 9, 2024.

**Graph facts in the prompt:**
- [G1] JB Pritzker [PERSON] --ADOPTS--> HB3773 [LEGAL_INSTRUMENT] (2024-10-22)

**Text-only answer** (citation coverage 1.0):

The individual who signed HB3773 into law in August 2024 was Illinois Governor JB Pritzker [S1](https://www.lawandtheworkplace.com/author/stevenpearlman/page/4/).

**Graph-enhanced answer** (citation coverage 1.0, cited: ['S1', 'G1']):

The individual who signed HB3773 into law in August 2024 was Illinois Governor JB Pritzker [S1](https://www.lawandtheworkplace.com/author/stevenpearlman/page/4/)[G1].

*11 s for both answers*

### temporal: On what date did Illinois Governor JB Pritzker sign HB3773 into law?
**Reference answer:** Illinois Governor JB Pritzker signed HB3773 into law on August 9, 2024.

**Graph facts in the prompt:**
- [G1] JB Pritzker [PERSON] --ADOPTS--> HB3773 [LEGAL_INSTRUMENT] (2024-10-22)

**Text-only answer** (citation coverage 1.0):

Illinois Governor JB Pritzker signed HB3773 into law on August 9, 2024 [S1](https://www.lawandtheworkplace.com/author/stevenpearlman/page/4/).

**Graph-enhanced answer** (citation coverage 0.667, cited: ['S1']):

Illinois Governor JB Pritzker signed HB3773 into law on August 9, 2024 [S1](https://www.lawandtheworkplace.com/author/stevenpearlman/page/4/). This information is also supported by the graph fact G1, which indicates that JB Pritzker adopted HB3773 on 2024-10-22, though this date appears to be the publication date of the source rather than the signing date. The correct signing date is explicitly stated in [S1](https://www.lawandtheworkplace.com/author/stevenpearlman/page/4/).

*14 s for both answers*

### factual: According to the report, what is the purpose of the Artificial Intelligence Act?
**Reference answer:** The Artificial Intelligence Act aims to regulate the use of artificial intelligence within the EU. It ensures that AI systems, including those used in law enforcement, comply with human rights standards and do not perpetuate biases or infringe upon individual freedoms.

**Graph facts in the prompt:**
- [G1] European Union [PUBLIC_AUTHORITY] --ADOPTS--> EU AI Act [LEGAL_INSTRUMENT] (2026-05-07)
- [G2] European Commission [PUBLIC_AUTHORITY] --PROPOSES--> EU AI Act [LEGAL_INSTRUMENT] (2025-01-22)
- [G3] European Artificial Intelligence Board [PUBLIC_AUTHORITY] --ENFORCES--> EU AI Act [LEGAL_INSTRUMENT] (2026-01-27)

**Text-only answer** (citation coverage 1.0):

The purpose of the Artificial Intelligence Act, according to the report, is to enhance the development of AI and support its ethical use through a comprehensive legal framework [S1](https://www.bakermckenzie.com/en/insight/publications/resources/product-risk-radar-articles/eu-regulation-on-ai). Additionally, the AI Act aims to protect fundamental rights, democracy, the rule of law, and environmental sustainability by taking a risk-based approach [S6](https://www.kpmg.com/de/en/insights/digital-transformation/artificial-intelligence/the-meaning-of-the-eu-law-on-ai.html). The Act also seeks to ensure that AI systems, including those used in law enforcement, comply with human rights standards and do not perpetuate biases or infringe upon individual freedoms [S2](https://eutoday.net/common-challenges-in-cybercrime-2024/#respond ).

**Graph-enhanced answer** (citation coverage 1.0, cited: ['S1', 'S6', 'S2', 'S5', 'G1', 'G2', 'G3']):

The purpose of the Artificial Intelligence Act, according to the report, is to enhance the development of AI and support its ethical use through a comprehensive legal framework [S1](https://www.bakermckenzie.com/en/insight/publications/resources/product-risk-radar-articles/eu-regulation-on-ai). It aims to protect fundamental rights, democracy, the rule of law, and environmental sustainability by taking a risk-based approach [S6](https://www.kpmg.com/de/en/insights/digital-transformation/artificial-intelligence/the-meaning-of-the-eu-law-on-ai.html). The AI Act also seeks to ensure that AI systems, including those used in law enforcement, comply with human rights standards and do not perpetuate biases or infringe upon individual freedoms [S2](https://eutoday.net/common-challenges-in-cybercrime-2024/#respond ). 

The AI Act entered a new enforcement phase on 2 August 2026, giving the European Commission powers to police obligations on general-purpose AI models and activating transparency rules for several kinds of AI system and synthetic content [S5](https://eutoday.net/eu-ai-act-enforcement-transparency-rules/). The legislation has been phased in since 2025, with some chapters becoming applicable as early as 2 February 2025 [S1](https://www.bakermckenzie.com/en/insight/publications/resources/product-risk-radar-articles/eu-regulation-on-ai). 

The AI Act was adopted by the European Union on 2026-05-07 [G1], proposed by the European Commission on 2025-01-22 [G2], and enforced by the European Artificial Intelligence Board on 2026-01-27 [G3].

*26 s for both answers*

In [9]:
from anatoolbox.enrich.synthesize.synthesize_answer import build_user_prompt

# 1. Graph facts only for questions about an action or an actor (the graph contains only actions)
ACTION_QUESTION_PATTERN = re.compile(
    r"\b(?:who|which (?:person|individual|company|companies|organi[sz]ations?|authority|authorities|agency|"
    r"regulators?|states?|countr(?:y|ies)|governments?|governor|lawmakers?|senators?))\b"
    r"|\b(?:propos\w*|introduc\w*|draft\w*|pass(?:ed|es)?|adopt\w*|enact\w*|sign\w*|approv\w*|enforc\w*|"
    r"fine[sd]?|investigat\w*|prob(?:e|ed|es|ing)|ban(?:s|ned)?|su(?:e|ed|es|ing)|lawsuits?|"
    r"compl(?:y|ies|ied|iance)|commit\w*|endors\w*|back(?:ed|s)?|support\w*|oppos\w*|criticiz\w*|"
    r"criticis\w*|challeng\w*|veto\w*|block\w*|revok\w*|repeal\w*)\b",
    re.IGNORECASE,
)


def graph_evidence_for_answer(question):
    """Relevant facts, but only for questions that ask about an action or an actor."""
    return graph_evidence(question) if ACTION_QUESTION_PATTERN.search(question) else []


# 2. Facts in the prompt: "reported on" date, validation status and evidence quote
FACT_INTRO = (
    "Graph facts from our knowledge graph. Each fact was extracted from a news passage, which is quoted "
    "as evidence. The facts carry no event dates: take all dates from the sources only. "
    "Use a fact only if it answers part of the question, and cite it with its label in brackets, e.g. [G1]."
)


def fact_line(fact):
    verb = PREDICATE_WORDS.get(fact["relation"], fact["relation"])
    return (f"{fact['subject']} {verb} {fact['object']} ({fact.get('validation_status')}). "
            f"Evidence: \"{fact.get('evidence') or ''}\"")


class SynthesizeAnswerGraph(SynthesizeAnswerOllama):
    """Graph-enhanced answers: facts with their reporting date, status and evidence quote."""
    tool_name = "synthesize_answer_ollama_graph"

    def user_prompt(self, question, sources, settings, facts=()):
        prompt = build_user_prompt(question, sources, instructions=settings["instructions"])
        if not facts:
            return prompt
        head = prompt.rsplit("\n\n", 1)[0]          # without the final instruction
        block = FACT_INTRO + "\n\n" + "\n".join(f"[{f['label']}] {fact_line(f)}" for f in facts)
        return f"{head}\n\n{block}\n\nAnswer the question, citing sources and graph facts by label."


synthesize_graph = SynthesizeAnswerGraph()


def answer_with_graph_v2(q):
    facts = graph_evidence_for_answer(q["question"])
    return answer(q, rag_passages_for(q["question"]), tool=synthesize_graph, facts=facts, max_facts=MAX_FACTS)


# 3. Coverage and the same three questions again
with_facts_v1 = sum(bool(graph_evidence(q["question"])) for q in EVAL_SET)
with_facts_v2 = sum(bool(graph_evidence_for_answer(q["question"])) for q in EVAL_SET)
print(f"Questions with graph evidence: {with_facts_v1} -> {with_facts_v2} (only action questions)")
display(pd.DataFrame([{"type": q["type"], "with_facts": bool(graph_evidence_for_answer(q["question"]))}
                      for q in EVAL_SET]).groupby("type")["with_facts"].agg(["sum", "mean"]).round(2))

print("\nExample prompt block:")
example_facts = graph_evidence_for_answer(demo[1]["question"])
print(FACT_INTRO + "\n" + "\n".join(f"[G{i}] {fact_line(f)}" for i, f in enumerate(example_facts, 1)))

for q in demo:
    result = answer_with_graph_v2(q)
    facts_shown = "\n".join(f"- [{f['label']}] {describe_fact(f)}" for f in result["facts"]) or "- (none)"
    display(Markdown(
        f"### {q['type']}: {q['question']}\n"
        f"**Reference answer:** {q['reference_answer']}\n\n"
        f"**Graph facts in the prompt:**\n{facts_shown}\n\n"
        f"**Graph-enhanced answer v2** (citation coverage {result['citation_coverage']}, cited: {result['cited']}):\n\n"
        f"{result['answer_markdown']}"
    ))

Questions with graph evidence: 24 -> 15 (only action questions)


,sum,mean
type,,
analytical,0,0.00
factual,1,0.03
relational,9,0.69
temporal,5,0.24



Example prompt block:
Graph facts from our knowledge graph. Each fact was extracted from a news passage, which is quoted as evidence. The facts carry no event dates: take all dates from the sources only. Use a fact only if it answers part of the question, and cite it with its label in brackets, e.g. [G1].
[G1] JB Pritzker adopted HB3773 (unreviewed). Evidence: "Illinois Governor JB Pritzker signed into law HB3773, a bill amending the Illinois Human Rights Act to address employers' use of artificial intelligence (AI)."


### relational: Which individual signed HB3773 into law in August 2024?
**Reference answer:** Illinois Governor JB Pritzker signed HB3773 into law on August 9, 2024.

**Graph facts in the prompt:**
- [G1] JB Pritzker [PERSON] --ADOPTS--> HB3773 [LEGAL_INSTRUMENT] (2024-10-22)

**Graph-enhanced answer v2** (citation coverage 1.0, cited: ['S1', 'G1']):

In August 2024, Illinois Governor JB Pritzker signed HB3773 into law [S1](https://www.lawandtheworkplace.com/author/stevenpearlman/page/4/). This bill amends the Illinois Human Rights Act to address employers' use of artificial intelligence (AI) [G1].

### temporal: On what date did Illinois Governor JB Pritzker sign HB3773 into law?
**Reference answer:** Illinois Governor JB Pritzker signed HB3773 into law on August 9, 2024.

**Graph facts in the prompt:**
- [G1] JB Pritzker [PERSON] --ADOPTS--> HB3773 [LEGAL_INSTRUMENT] (2024-10-22)

**Graph-enhanced answer v2** (citation coverage 1.0, cited: ['S1', 'G1']):

Illinois Governor JB Pritzker signed HB3773 into law on August 9, 2024 [S1](https://www.lawandtheworkplace.com/author/stevenpearlman/page/4/). This date is supported by the fact that "On August 9, 2024, Illinois Governor JB Pritzker signed into law HB3773" [G1].

### factual: According to the report, what is the purpose of the Artificial Intelligence Act?
**Reference answer:** The Artificial Intelligence Act aims to regulate the use of artificial intelligence within the EU. It ensures that AI systems, including those used in law enforcement, comply with human rights standards and do not perpetuate biases or infringe upon individual freedoms.

**Graph facts in the prompt:**
- (none)

**Graph-enhanced answer v2** (citation coverage 1.0, cited: ['S1', 'S6', 'S2']):

The purpose of the Artificial Intelligence Act, according to the report, is to enhance the development of AI and support its ethical use through a comprehensive legal framework [S1](https://www.bakermckenzie.com/en/insight/publications/resources/product-risk-radar-articles/eu-regulation-on-ai). Additionally, the AI Act aims to protect fundamental rights, democracy, the rule of law, and environmental sustainability by taking a risk-based approach [S6](https://www.kpmg.com/de/en/insights/digital-transformation/artificial-intelligence/the-meaning-of-the-eu-law-on-ai.html). The Act also seeks to ensure that AI systems, including those used in law enforcement, comply with human rights standards and do not perpetuate biases or infringe upon individual freedoms [S2](https://eutoday.net/common-challenges-in-cybercrime-2024/#respond ).

In [10]:
# ===== 5.1 Judge model: gemma3:12b via Ollama =====
import json
from anatoolbox.analyze.score.score_rag_answer import ScoreRagAnswerTool, RESPONSE_SCHEMA

JUDGE_MODEL = "gemma3:12b"   # the model that generated the Q&A pairs, different from the RAG model

if JUDGE_MODEL not in [m.model for m in ollama.list().models]:
    print(f"Downloading {JUDGE_MODEL} (about 8 GB) ...")
    ollama.pull(JUDGE_MODEL)

OLLAMA_JUDGE_OPTIONS = {"temperature": 0, "seed": OLLAMA_RAG_OPTIONS["seed"], "num_ctx": 12288}


class ScoreRagAnswerOllama(ScoreRagAnswerTool):
    """score_rag_answer through Ollama's own API (large context window).

    The judge sees the graph facts exactly as the answering model saw them: without dates, with evidence.
    """
    tool_name = "score_rag_answer_ollama"

    def judge(self, system, user, settings, context):
        response = ollama.chat(
            model=settings["model"],
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            format=RESPONSE_SCHEMA,
            options=OLLAMA_JUDGE_OPTIONS,
        )
        return json.loads(response.message.content)

    def user_prompt(self, question, answer, sources, settings, facts=()):
        prompt = super().user_prompt(question, answer, sources, settings, facts=())
        if not facts:
            return prompt
        head, tail = prompt.split("\n\nReference answer: ", 1)
        block = ("Graph facts (no event dates; the evidence is quoted from a news passage):\n\n"
                 + "\n".join(f"[{f['label']}] {fact_line(f)}" for f in facts))
        return f"{head}\n\n{block}\n\nReference answer: {tail}"


judge = ScoreRagAnswerOllama()


def judge_answer(q, result):
    return judge.run({"answer": result, "reference_answer": q["reference_answer"], "model": JUDGE_MODEL},
                     context=ctx)


# Test on the relational and the temporal demo question
for q in demo[:2]:
    for label, answer_fn in [("text-only", answer_text_only), ("graph v2", answer_with_graph_v2)]:
        started = time.perf_counter()
        result = answer_fn(q)
        judged = judge_answer(q, result)
        print(f"\n{q['type']} | {label} | {time.perf_counter() - started:.0f} s")
        print("  scores:", judged["scores"])
        for criterion, rationale in judged["rationales"].items():
            print(f"  {criterion}: {rationale}")
        if judged["unsupported_claims"]:
            print("  unsupported claims:", judged["unsupported_claims"])
        if judged["review_flags"]:
            print("  review flags:", judged["review_flags"])


relational | text-only | 30 s
  scores: {'faithfulness': 5, 'relevance': 5, 'context_grounding': 5, 'temporal_grounding': 5, 'correctness': 5}
  faithfulness: The answer accurately reflects the information provided in S1, stating that Illinois Governor JB Pritzker signed HB3773 into law.
  relevance: The answer directly and completely addresses the question by identifying the individual who signed HB3773.
  context_grounding: The answer appropriately uses the provided evidence (S1) to support its claim and cites it correctly.
  temporal_grounding: The answer correctly states the date (August 9, 2024) as provided in S1, aligning with the question's timeframe.
  correctness: The answer perfectly matches the reference answer, providing the correct individual and date.

relational | graph v2 | 31 s
  scores: {'faithfulness': 5, 'relevance': 5, 'context_grounding': 5, 'temporal_grounding': 5, 'correctness': 5}
  faithfulness: The answer accurately reflects the information provided in S1 an

In [11]:
# ===== 5.2 Full evaluation: text-only vs graph-enhanced (v2) =====
EVAL_FOLDER = os.path.join(STAGE3_FOLDER, "eval")
os.makedirs(EVAL_FOLDER, exist_ok=True)
ANSWERS_FILE = os.path.join(EVAL_FOLDER, "rag_answers.jsonl")
JUDGMENTS_FILE = os.path.join(EVAL_FOLDER, "rag_judgments.jsonl")
CRITERIA = ["faithfulness", "relevance", "context_grounding", "temporal_grounding", "correctness"]
TEXT_ONLY, GRAPH = "text-only", "graph-enhanced v2"
QUESTIONS = {q["id"]: q for q in EVAL_SET}


def load_jsonl(path):
    if not os.path.exists(path):
        return {}
    with open(path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    return {(r["configuration"], r["id"]): r for r in rows}


def append_jsonl(path, row):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")


def run_phase(path, work, label):
    """Run `work(configuration, question, done)` for all open items; results are saved one by one."""
    done = load_jsonl(path)
    items = [(c, q) for c in (TEXT_ONLY, GRAPH) for q in EVAL_SET]   # text-only first (needed for reuse)
    todo = [(c, q) for c, q in items if (c, q["id"]) not in done]
    started, failed = time.perf_counter(), []
    for i, (c, q) in enumerate(todo, 1):
        try:
            row = {"configuration": c, "id": q["id"], **work(c, q, done)}
        except Exception as error:   # one failed call costs one question, not the run
            failed.append(f"{c}/{q['id']}: {type(error).__name__}")
            continue
        append_jsonl(path, row)
        done[(c, q["id"])] = row
        if i % 10 == 0:
            print(f"  {label}: {i}/{len(todo)} ({time.perf_counter() - started:.0f} s)")
    print(f"{label}: {len(todo) - len(failed)} new, {len(done)} in total, "
          f"{len(failed)} failed {failed[:3]} ({time.perf_counter() - started:.0f} s)")
    return done


def make_answer(c, q, done):
    if c == TEXT_ONLY:
        return {"result": answer_text_only(q), "reused": False}
    if not graph_evidence_for_answer(q["question"]):   # no facts → identical prompt → same answer
        return {"result": done[(TEXT_ONLY, q["id"])]["result"], "reused": True}
    return {"result": answer_with_graph_v2(q), "reused": False}


def make_judgment(c, q, done):
    answer_row = answers[(c, q["id"])]
    if answer_row["reused"]:
        text_only = done[(TEXT_ONLY, q["id"])]
        return {k: text_only[k] for k in ("scores", "rationales", "unsupported_claims", "review_flags")}
    judged = judge_answer(q, answer_row["result"])
    return {k: judged[k] for k in ("scores", "rationales", "unsupported_claims", "review_flags")}


# Phase 1: answers (qwen3:8b), phase 2: judgments (gemma3:12b)
answers = run_phase(ANSWERS_FILE, make_answer, "answers")
judgments = run_phase(JUDGMENTS_FILE, make_judgment, "judgments")

# Results table
rows = []
for (c, qid), j in judgments.items():
    q, result = QUESTIONS[qid], answers[(c, qid)]["result"]
    rows.append({
        "configuration": c, "id": qid, "type": q["type"], "question": q["question"], **j["scores"],
        "citation_coverage": result["citation_coverage"],
        "graph_facts": len(result.get("facts") or []),
        "unsupported_claims": len(j["unsupported_claims"]),
        "review_flags": "; ".join(j["review_flags"]),
        "reused": answers[(c, qid)]["reused"],
        "answer": result["answer"],
    })
RAG_RESULTS = pd.DataFrame(rows)
RAG_RESULTS.to_csv(os.path.join(EVAL_FOLDER, "rag_results.csv"), index=False)
METRICS = CRITERIA + ["citation_coverage", "unsupported_claims"]

print("\nAll questions:")
display(RAG_RESULTS.groupby("configuration")[METRICS].mean().round(2))

print("By question type:")
display(RAG_RESULTS.pivot_table(index="type", columns="configuration", values=CRITERIA, aggfunc="mean").round(2))

has_facts = RAG_RESULTS["id"].isin(RAG_RESULTS.loc[RAG_RESULTS["graph_facts"] > 0, "id"])
print(f"Only the {RAG_RESULTS.loc[has_facts, 'id'].nunique()} questions with graph facts:")
display(RAG_RESULTS[has_facts].groupby("configuration")[METRICS].mean().round(2))

paired = RAG_RESULTS[has_facts].pivot(index="id", columns="configuration", values=CRITERIA + ["unsupported_claims"])
difference = paired.xs(GRAPH, axis=1, level=1) - paired.xs(TEXT_ONLY, axis=1, level=1)
print("Graph vs text-only per question (better / same / worse):")
display(pd.DataFrame({m: [(difference[m] > 0).sum(), (difference[m] == 0).sum(), (difference[m] < 0).sum()]
                      for m in CRITERIA}, index=["better", "same", "worse"]))

flagged = RAG_RESULTS[(RAG_RESULTS["review_flags"] != "") & ~RAG_RESULTS["reused"]]
print(f"{len(flagged)} judgments flagged for review")

  answers: 10/206 (75 s)
  answers: 20/206 (149 s)
  answers: 30/206 (226 s)
  answers: 40/206 (299 s)
  answers: 50/206 (368 s)
  answers: 60/206 (439 s)
  answers: 70/206 (512 s)
  answers: 80/206 (585 s)
  answers: 90/206 (657 s)
  answers: 100/206 (725 s)
  answers: 110/206 (757 s)
  answers: 120/206 (764 s)
  answers: 130/206 (778 s)
  answers: 140/206 (796 s)
  answers: 150/206 (796 s)
  answers: 160/206 (813 s)
  answers: 170/206 (820 s)
  answers: 180/206 (840 s)
  answers: 190/206 (840 s)
  answers: 200/206 (847 s)
answers: 206 new, 206 in total, 0 failed [] (849 s)
  judgments: 10/206 (185 s)
  judgments: 20/206 (387 s)
  judgments: 30/206 (593 s)
  judgments: 40/206 (798 s)
  judgments: 50/206 (1005 s)
  judgments: 60/206 (1200 s)
  judgments: 70/206 (1393 s)
  judgments: 80/206 (1598 s)
  judgments: 90/206 (1788 s)
  judgments: 100/206 (1991 s)
  judgments: 110/206 (2068 s)
  judgments: 120/206 (2089 s)
  judgments: 130/206 (2128 s)
  judgments: 140/206 (2171 s)
  judgments

,faithfulness,relevance,context_grounding,temporal_grounding,correctness,citation_coverage,unsupported_claims
configuration,,,,,,,
graph-enhanced v2,4.83,4.98,4.96,4.92,4.83,0.82,0.09
text-only,4.83,4.98,4.95,4.92,4.81,0.81,0.09


By question type:


context_grounding                 correctness            \
configuration graph-enhanced v2 text-only graph-enhanced v2 text-only   
type                                                                    
analytical                 4.88      4.88              4.73      4.73   
factual                    5.00      5.00              4.89      4.89   
relational                 5.00      4.92              5.00      4.85   
temporal                   5.00      5.00              4.76      4.76   

                   faithfulness                   relevance            \
configuration graph-enhanced v2 text-only graph-enhanced v2 text-only   
type                                                                    
analytical                 4.67      4.67              4.94      4.94   
factual                    4.92      4.92              5.00      5.00   
relational                 5.00      4.92              5.00      5.00   
temporal                   4.86      4.86              5.00      5.00   

              temporal_grounding            
configuration  graph-enhanced v2 text-only  
type                                        
analytical                  4.56      4.56  
factual                     5.00      5.00  
relational                  5.00      5.00  
temporal                    5.00      5.00

Only the 15 questions with graph facts:


,faithfulness,relevance,context_grounding,temporal_grounding,correctness,citation_coverage,unsupported_claims
configuration,,,,,,,
graph-enhanced v2,5.00,5.0,5.00,5.0,5.00,0.91,0.07
text-only,4.93,5.0,4.93,5.0,4.87,0.83,0.07


Graph vs text-only per question (better / same / worse):


,faithfulness,relevance,context_grounding,temporal_grounding,correctness
better,1,0,1,0,1
same,14,15,14,11,14
worse,0,0,0,0,0


1 judgments flagged for review


In [12]:
# ===== 5.3 Error analysis: the changed question, the flagged judgment and the weakest answers =====
def show_judged(c, qid):
    j, a = judgments[(c, qid)], answers[(c, qid)]["result"]
    print(f"\n[{c}] scores {j['scores']}\n{a['answer']}")
    print("  faithfulness:", j["rationales"]["faithfulness"])
    print("  correctness:", j["rationales"]["correctness"])
    if j["unsupported_claims"]:
        print("  unsupported claims:", j["unsupported_claims"])


print("=== Questions where graph and text-only differ ===")
for qid in difference[(difference != 0).any(axis=1)].index:
    q = QUESTIONS[qid]
    print(f"\n### {q['type']}: {q['question']}\nReference: {q['reference_answer']}")
    for c in (TEXT_ONLY, GRAPH):
        show_judged(c, qid)

print("\n=== Flagged judgments ===")
for _, row in flagged.iterrows():
    print(f"\n{row['configuration']} | {row['type']}: {row['question']}\nflags: {row['review_flags']}")
    show_judged(row["configuration"], row["id"])

text_only_results = RAG_RESULTS[RAG_RESULTS["configuration"] == TEXT_ONLY]
weak = text_only_results[text_only_results[["faithfulness", "correctness"]].min(axis=1) <= 3]
print(f"\n=== {len(weak)} text-only answers with faithfulness or correctness <= 3 ===")
display(weak[["id", "type", "question", "faithfulness", "correctness", "temporal_grounding", "unsupported_claims"]])
for qid in weak["id"]:
    q = QUESTIONS[qid]
    print(f"\n### {q['type']}: {q['question']}\nReference: {q['reference_answer']}")
    show_judged(TEXT_ONLY, qid)

=== Questions where graph and text-only differ ===

### relational: Which organization requested that Virginia Governor Glenn Youngkin veto House Bill 2094?
Reference: The U.S. Chamber of Commerce requested that Virginia Governor Glenn Youngkin veto House Bill 2094 in a letter sent last month.

[text-only] scores {'faithfulness': 5, 'relevance': 5, 'context_grounding': 5, 'temporal_grounding': None, 'correctness': 5}
The U.S. Chamber of Commerce requested that Virginia Governor Glenn Youngkin veto House Bill 2094 [S4].
  faithfulness: The answer accurately reflects the information provided in source [S4] and does not introduce any unsupported claims.
  correctness: The answer aligns perfectly with the reference answer and presents the correct information.

[graph-enhanced v2] scores {'faithfulness': 5, 'relevance': 5, 'context_grounding': 5, 'temporal_grounding': None, 'correctness': 5}
The U.S. Chamber of Commerce requested that Virginia Governor Glenn Youngkin veto House Bill 2094 [S

,id,type,question,faithfulness,correctness,temporal_grounding,unsupported_claims
8,qa_10129_002_0,factual,"According to Meta, what EU regulation delayed ...",4,3,NaN,0
43,qa_15702_007_1,analytical,Why did the Justice Department dismiss the cas...,1,2,3.0,2
47,qa_16403_001_0,relational,Which organizations and individuals supported ...,4,3,5.0,1
62,qa_18333_007_2,temporal,When and where was Nvidia's GTC developer's ev...,3,3,5.0,2
79,qa_21900_000_1,analytical,"According to the August 13, 2025, article in T...",3,3,NaN,1
80,qa_21900_000_2,temporal,When did the main provision of the Online Safe...,5,3,5.0,0



### factual: According to Meta, what EU regulation delayed the launch of its glasses in EU countries?
Reference: According to Meta, GDPR, the EU's data privacy framework introduced in 2018, delayed the launch of its glasses in EU countries.

[text-only] scores {'faithfulness': 4, 'relevance': 5, 'context_grounding': 5, 'temporal_grounding': None, 'correctness': 3}
According to Meta, the EU regulation that delayed the launch of its glasses in EU countries was the AI Act, along with GDPR [S1]. Meta cited issues surrounding its use of Instagram and Facebook user data to train its AI models as a key factor in the delay [S1]. Additionally, Meta mentioned the "complex regulatory system" in Europe as a reason for the delay in rolling out AI features for its Ray-Ban Meta glasses in some European countries [S2].
  faithfulness: The answer accurately reflects information from the sources, but it combines information from multiple sources and slightly rephrases it, which could be interpreted as 

In [13]:
# ===== 6.1 Dates in the questions =====
import calendar
from datetime import date, timedelta

MONTHS = {m.lower(): i for i, m in enumerate(calendar.month_name) if m}
MONTH_NAMES = "|".join(MONTHS)
FULL_DATE = re.compile(rf"\b({MONTH_NAMES})\s+(\d{{1,2}}),?\s+(20\d\d)\b"
                       rf"|\b(\d{{1,2}})\s+({MONTH_NAMES})\s+(20\d\d)\b"
                       rf"|\b(20\d\d)-(\d\d)-(\d\d)\b", re.IGNORECASE)
MONTH_YEAR = re.compile(rf"\b({MONTH_NAMES})\s+(?:of\s+)?(20\d\d)\b", re.IGNORECASE)
YEAR = re.compile(r"\b(?:in|during|since|by|of)\s+(20\d\d)\b", re.IGNORECASE)
PUBLICATION_CUE = re.compile(r"\b(?:article|report|press release|post|story|published|according to)\b", re.IGNORECASE)
PUBLICATION_SLACK_DAYS = 3    # the article itself: its date ± 3 days
EVENT_LAG_DAYS = 90           # an event: reported from its period up to 90 days later


def question_period(question):
    """(kind, date_from, date_to) for the first date in the question, or None."""
    m = FULL_DATE.search(question)
    if m:
        g = m.groups()
        try:
            if g[0]:
                d = date(int(g[2]), MONTHS[g[0].lower()], int(g[1]))
            elif g[3]:
                d = date(int(g[5]), MONTHS[g[4].lower()], int(g[3]))
            else:
                d = date(int(g[6]), int(g[7]), int(g[8]))
        except ValueError:
            return None
        if PUBLICATION_CUE.search(question):
            return "publication", d - timedelta(PUBLICATION_SLACK_DAYS), d + timedelta(PUBLICATION_SLACK_DAYS)
        return "event", d, d + timedelta(EVENT_LAG_DAYS)
    m = MONTH_YEAR.search(question)
    if m:
        year, month = int(m.group(2)), MONTHS[m.group(1).lower()]
        last_day = date(year, month, calendar.monthrange(year, month)[1])
        return "event", date(year, month, 1), last_day + timedelta(EVENT_LAG_DAYS)
    m = YEAR.search(question)
    if m:
        year = int(m.group(1))
        return "event", date(year, 1, 1), date(year, 12, 31) + timedelta(EVENT_LAG_DAYS)
    return None


# How many questions name a date, and does the source article fall into the derived period?
article_dates = rag_records.groupby("source_id")["date"].first().astype(str).str[:10]
rows = []
for q in EVAL_SET:
    period = question_period(q["question"])
    if period is None:
        continue
    kind, date_from, date_to = period
    source_dates = [article_dates[s] for s in q["source_ids"] if s in article_dates.index]
    rows.append({
        "id": q["id"], "type": q["type"], "kind": kind,
        "date_from": date_from.isoformat(), "date_to": date_to.isoformat(),
        "source_date": source_dates[0] if source_dates else None,
        "source_in_period": any(date_from.isoformat() <= s <= date_to.isoformat() for s in source_dates),
        "text_only_hit@5": base_per_question.loc[q["id"], "hit@5"],
        "question": q["question"][:90],
    })
dated = pd.DataFrame(rows)

print(f"{len(dated)} of {len(EVAL_SET)} questions name a date")
display(dated.groupby("type").size().rename("questions").to_frame())
print("Is the source article in the derived period?")
display(dated.groupby("kind")["source_in_period"].agg(["sum", "size", "mean"]).round(2))
print("Text-only hit@5 on these questions:", round(dated["text_only_hit@5"].mean(), 3))
print("\nQuestions whose source article lies outside the period:")
display(dated.loc[~dated["source_in_period"], ["kind", "date_from", "date_to", "source_date", "question"]])

30 of 103 questions name a date


,questions
type,
analytical,14
factual,13
relational,3


Is the source article in the derived period?


,sum,size,mean
kind,,,
event,16,21,0.76
publication,9,9,1.00


Text-only hit@5 on these questions: 0.867

Questions whose source article lies outside the period:


,kind,date_from,date_to,source_date,question
0,event,2023-09-01,2023-12-29,2024-10-01,"According to the lawsuit, what two lists was G..."
18,event,2022-11-01,2023-02-28,2025-07-08,"Which two open-source LLMs, Meta's Llama 2 and..."
19,event,2022-11-01,2023-02-28,2025-07-08,Why did Kolter believe that breaking open-sour...
22,event,2025-08-01,2025-11-29,2025-07-23,"Who will the AI Act apply to, according to the..."
23,event,2025-02-01,2025-05-29,2025-07-24,According to an executive order released in Fe...


In [14]:
# ===== 6.2 Time-aware retrieval: before and after =====
def retrieve_time_aware(question, use_events=True):
    """Publication date → hard filter; event date → candidates from the period added to the usual pool."""
    period = question_period(question)
    if period is None or (period[0] == "event" and not use_events):
        return retrieve_for_rag(question)
    kind, date_from, date_to = period
    in_period = search(question, strategy="hybrid", date_from=date_from.isoformat(), date_to=date_to.isoformat())
    if kind == "publication":
        candidates = in_period if in_period["passages"] else search(question, strategy="hybrid")
    else:
        base = search(question, strategy="hybrid")
        seen, merged = set(), []
        for p in base["passages"] + in_period["passages"]:
            if p["id"] not in seen:
                seen.add(p["id"])
                merged.append(p)
        candidates = {**base, "passages": merged}
    return rerank.run({"input": candidates, "keep": 30}, context=ctx)


def metrics_on(question_ids, retrieve_fn):
    runs = [{"id": qid, "question": questions_by_id[qid]["question"],
             "relevant": questions_by_id[qid]["source_ids"], "retrieved": retrieve_fn(questions_by_id[qid]["question"])}
            for qid in question_ids]
    return pd.DataFrame(retrieval_metrics.run({"results": runs, "k": [1, 3, 5, 10]}, context=ctx)["per_question"]).set_index("id")


# Only the 30 dated questions change; all others keep their baseline values
dated_ids = list(dated["id"])
METRIC_COLUMNS = list(base_per_question.select_dtypes("number").columns)
VARIANTS = {
    "text-only (baseline)": None,
    "+ publication filter": lambda question: retrieve_time_aware(question, use_events=False),
    "+ publication filter + event candidates": lambda question: retrieve_time_aware(question, use_events=True),
}
time_results = {}
for label, retrieve_fn in VARIANTS.items():
    combined = base_per_question[METRIC_COLUMNS].copy()
    if retrieve_fn is not None:
        started = time.perf_counter()
        changed = metrics_on(dated_ids, retrieve_fn)
        combined.loc[changed.index, METRIC_COLUMNS] = changed[METRIC_COLUMNS]
        print(f"{label}: {time.perf_counter() - started:.0f} s")
    time_results[label] = combined

print("\nAll 103 questions:")
display(pd.DataFrame({label: df.mean() for label, df in time_results.items()}).T.round(3))
print("Only the 30 questions with a date:")
display(pd.DataFrame({label: df.loc[dated_ids].mean() for label, df in time_results.items()}).T.round(3))
kind_of = dated.set_index("id")["kind"]
print("hit@5 by kind of date:")
display(pd.DataFrame({label: df.loc[dated_ids, "hit@5"].groupby(kind_of).mean()
                      for label, df in time_results.items()}).round(3))

final = "+ publication filter + event candidates"
delta = time_results[final].loc[dated_ids, "hit@5"] - time_results["text-only (baseline)"].loc[dated_ids, "hit@5"]
print("Questions whose hit@5 changes with the final variant:")
display(pd.DataFrame([{"change": int(delta[qid]), "kind": kind_of[qid], "question": questions_by_id[qid]["question"][:100]}
                      for qid in delta.index[delta != 0]]))

+ publication filter: 15 s
+ publication filter + event candidates: 17 s

All 103 questions:


,retrieved,relevant,precision@1,recall@1,hit@1,precision@3,recall@3,hit@3,precision@5,recall@5,hit@5,precision@10,recall@10,hit@10,reciprocal_rank
text-only (baseline),20.359,1.0,0.728,0.728,0.728,0.298,0.893,0.893,0.183,0.913,0.913,0.092,0.922,0.922,0.812
+ publication filter,19.485,1.0,0.748,0.748,0.748,0.304,0.913,0.913,0.186,0.932,0.932,0.094,0.942,0.942,0.832
+ publication filter + event candidates,19.369,1.0,0.748,0.748,0.748,0.304,0.913,0.913,0.186,0.932,0.932,0.094,0.942,0.942,0.832


Only the 30 questions with a date:


,retrieved,relevant,precision@1,recall@1,hit@1,precision@3,recall@3,hit@3,precision@5,recall@5,hit@5,precision@10,recall@10,hit@10,reciprocal_rank
text-only (baseline),21.067,1.0,0.700,0.700,0.700,0.289,0.867,0.867,0.173,0.867,0.867,0.087,0.867,0.867,0.783
+ publication filter,18.067,1.0,0.767,0.767,0.767,0.311,0.933,0.933,0.187,0.933,0.933,0.093,0.933,0.933,0.850
+ publication filter + event candidates,17.667,1.0,0.767,0.767,0.767,0.311,0.933,0.933,0.187,0.933,0.933,0.093,0.933,0.933,0.852


hit@5 by kind of date:


,text-only (baseline),+ publication filter,+ publication filter + event candidates
kind,,,
event,0.905,0.905,0.905
publication,0.778,1.000,1.000


Questions whose hit@5 changes with the final variant:


,change,kind,question
0,1,publication,"According to the August 13, 2025, article in T..."
1,1,publication,"According to a report published on October 17,..."
